# Lab 3, part A: the policy

Lab 3 (A+B) Costs: $0.134

Scenario 1, a customer support resolution agent. It handles returns, billing disputes and
account questions against four backend tools, and it is measured on first-contact
resolution: resolve what it can, escalate what it must, and be right about which is which.

The exam's evidence for why this lab exists is one number. Told in its system prompt to
verify the customer first, a support agent skipped `get_customer` in 12% of production
cases and looked orders up on the name the customer typed, which misidentified accounts and
refunded the wrong people. Stronger wording moves that number. It never reaches zero,
because it still asks the model to choose to comply.

So this part builds the half that does not ask. Four tools, the failure shapes they return,
two gates that decide what may run, one pass that decides what the model gets to read, and
a handoff a human can act on without the transcript.

Part A makes no API calls. Every mechanism here is a plain async function, so all of it is
proven by calling it, offline and free. Part B hands the lot to an agent and watches what
changes.

In [ ]:
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

LAB = Path.cwd()
CODE = LAB.parent
if not (CODE / "pyproject.toml").exists():
    raise SystemExit(
        f"Run this notebook from its own folder inside labs/code. The working "
        f"directory is {LAB}, and {CODE / 'pyproject.toml'} is not there. "
        f"In JupyterLab the working directory follows the notebook, so open it "
        f"from the file browser rather than starting the kernel elsewhere."
    )

WORKSPACE = LAB / "workspace"
DATA = WORKSPACE / "data"
SUPPORT = WORKSPACE / "support"

NOW = datetime.now(timezone.utc)


def days_ago(n):
    return (NOW - timedelta(days=n)).replace(microsecond=0)


def unix(n):
    return int(days_ago(n).timestamp())


# get_customer answers with Unix timestamps. lookup_order answers with ISO 8601 and a
# numeric status code. Same concepts, two shapes: that is the problem section 5 removes.
CUSTOMERS = [
    {"customer_id": "CUST-0001", "name": "Ivan Petrov", "email": "ivan.petrov@example.com",
     "phone": "07700900001", "tier": "standard", "created": unix(420), "last_contact_at": unix(2)},
    {"customer_id": "CUST-0007", "name": "Ada Whitfield", "email": "ada.whitfield@example.com",
     "phone": "07700900007", "tier": "standard", "created": unix(300), "last_contact_at": unix(70)},
    {"customer_id": "CUST-0011", "name": "Marek Nowak", "email": "marek.nowak@example.com",
     "phone": "07700900011", "tier": "premium", "created": unix(210), "last_contact_at": unix(9)},
    {"customer_id": "CUST-0017", "name": "Leah Mensah", "email": "leah.mensah@example.com",
     "phone": "07700900017", "tier": "standard", "created": unix(150), "last_contact_at": unix(4)},
    {"customer_id": "CUST-0023", "name": "Sam Okafor", "email": "sam.okafor@example.com",
     "phone": "07700900023", "tier": "standard", "created": unix(95), "last_contact_at": unix(30)},
    {"customer_id": "CUST-0031", "name": "Sam Okafor", "email": "s.okafor@example.net",
     "phone": "07700900031", "tier": "premium", "created": unix(60), "last_contact_at": unix(11)},
]

STATUS_CODES = {"placed": 10, "dispatched": 20, "delivered": 30, "returned": 40, "cancelled": 50}


def order(order_id, customer_id, total_pence, delivered, item, sku, shipment, invoice):
    """A verbose backend record. Thirty-three fields arrive; five decide a refund."""
    return {
        "order_id": order_id, "customer_id": customer_id,
        "status": STATUS_CODES["delivered"], "status_detail": "left with resident",
        "placed_at": days_ago(delivered + 3).isoformat(),
        "dispatched_at": days_ago(delivered + 1).isoformat(),
        "delivered_at": days_ago(delivered).isoformat(),
        "updated_at": days_ago(delivered).isoformat(),
        "currency": "GBP", "subtotal_pence": total_pence - 499, "shipping_pence": 499,
        "tax_pence": round(total_pence / 6), "total_pence": total_pence, "discount_pence": 0,
        "refunded_pence": 0, "payment_method": "card", "payment_reference": "PAY-" + order_id[-4:],
        "invoice_id": invoice, "shipment_id": shipment, "carrier": "Northbound Logistics",
        "tracking_url": "https://tracking.example.com/" + shipment, "warehouse": "LEE-2",
        "channel": "web", "locale": "en-GB", "item_sku": sku, "item_name": item,
        "item_quantity": 1, "item_unit_pence": total_pence - 499, "gift_wrap": False,
        "marketing_source": "organic", "weight_grams": 1450, "return_window_days": 30,
        "signature_required": False,
    }


ORDERS = [
    order("ORD-0003", "CUST-0001", 8999, 18, "Ridgeline walking boots", "SKU-441", "SHP-0004", "INV-0002"),
    order("ORD-0021", "CUST-0001", 65000, 10, "Aurora espresso machine", "SKU-902", "SHP-0022", "INV-0020"),
    order("ORD-0009", "CUST-0007", 12000, 67, "Cormorant rain shell", "SKU-118", "SHP-0010", "INV-0008"),
    order("ORD-0014", "CUST-0011", 21000, 6, "Harrow cast iron set", "SKU-655", "SHP-0015", "INV-0013"),
    order("ORD-0019", "CUST-0017", 6450, 8, "Tamar wool jumper", "SKU-207", "SHP-0020", "INV-0018"),
]

# INV-0002 is the duplicate charge on the Chapter 9 slide: the same amount, twice.
INVOICES = [
    {"invoice_id": "INV-0002", "customer_id": "CUST-0001", "order_id": "ORD-0003",
     "amount_pence": 4000, "charges": 2, "charged_at": days_ago(17).isoformat()},
    {"invoice_id": "INV-0020", "customer_id": "CUST-0001", "order_id": "ORD-0021",
     "amount_pence": 65000, "charges": 1, "charged_at": days_ago(11).isoformat()},
]

POLICY = '''
# Support policy

- Refunds may be processed within **30 days** of delivery. Outside that window, offer a
  replacement or store credit.
- Price adjustments apply to **our own site only**, when the price drops within 14 days of
  purchase.
- An agent may refund up to **500 pounds**. Anything above that goes to a human.
- Damaged or faulty goods are replaced free of charge inside the refund window.
'''

DATA.mkdir(parents=True, exist_ok=True)
SUPPORT.mkdir(parents=True, exist_ok=True)
for name, rows in (("customers", CUSTOMERS), ("orders", ORDERS), ("invoices", INVOICES)):
    (DATA / f"{name}.json").write_text(json.dumps(rows, indent=2) + chr(10), encoding="utf-8")
(WORKSPACE / "policy.md").write_text(POLICY.lstrip(), encoding="utf-8")

print(f"workspace at {WORKSPACE}")
print(f"  data/customers.json   {len(CUSTOMERS)} customers, two of them sharing a name")
print(f"  data/orders.json      {len(ORDERS)} orders, {len(ORDERS[0])} fields each")
print(f"  data/invoices.json    {len(INVOICES)} invoices")
print("  policy.md             30 day window, own site only, 500 pound agent limit")
print()
print("The policy is silent on competitor price matching. That silence is deliberate.")

## 2. Four tools, and what each returns when it fails

The four the scenario names: `get_customer`, `lookup_order`, `process_refund` and
`escalate_to_human`. They run in-process, inside this Python session, through the Agent
SDK's own MCP server. No subprocess, no transport, no `.mcp.json`. The key they are
registered under is what names them, so `support` gives us
`mcp__support__get_customer` and its three siblings: the strings every rule in this lab
keys on.

Half of a tool's job is answering. The other half is failing in a shape the agent can act
on. Four categories, and one case that is not a failure at all:

| Category | Retryable | What the agent should do | Here |
|---|---|---|---|
| transient | yes | retry, with a delay | the orders API times out |
| validation | yes, after the input changes | fix the arguments, then retry | no account matches that identifier |
| business | no | explain it to the person, in their terms | the order is outside the refund window |
| permission | no | escalate | the agent is not authorised |

The fifth shape is a query that ran correctly and matched nothing, or matched more than one
thing. `get_customer` finding two accounts is a **success**, not an error. Hiding that
behind a ranking that returns the likeliest match is the failure this lab is built to
avoid: the agent must ask for another identifier, and it can only do that if the tool tells
it the truth.

One spelling note, because both appear in the course. On the MCP wire the flag is `isError`,
which is the exam's term. The SDK's `@tool` decorator forwards `is_error`, so that is what
this file returns. Lab 1's FastMCP tools had to **raise**, because a returned dictionary was
reported as a success; an SDK tool returns the flag instead.

In [ ]:
TOOLS_PY = '''
"""Four support tools, served in-process by the Agent SDK."""
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

from claude_agent_sdk import create_sdk_mcp_server, tool
from pydantic import BaseModel, Field, ValidationError, field_validator

DATA = Path(__file__).resolve().parent.parent / "data"
REFUND_WINDOW_DAYS = 30
FLAKY = {"ORD-0009"}          # the orders API times out on this one, once
_timed_out = set()


def reset():
    """Put the transient failure back, so the notebook can be rerun."""
    _timed_out.clear()


def load(name):
    return json.loads((DATA / (name + ".json")).read_text(encoding="utf-8"))


def ok(payload):
    return {"content": [{"type": "text", "text": json.dumps(payload, indent=2)}]}


def failure(category, message, retryable, **extra):
    """A failure the agent can act on: the category, whether a retry can help, and
    what to do instead. is_error is what the SDK forwards; isError is the wire name."""
    payload = {"errorCategory": category, "isRetryable": retryable, "message": message}
    payload.update(extra)
    return {"content": [{"type": "text", "text": json.dumps(payload, indent=2)}],
            "is_error": True}


def mask(value, keep=2):
    head, _, tail = value.partition("@")
    hidden = head[:keep] + "*" * max(len(head) - keep, 1)
    return hidden + ("@" + tail if tail else "")


@tool(
    "get_customer",
    "Verify who you are talking to, from an email address, a phone number or a customer "
    "id. Returns a verified customer id only when exactly one account matches. When more "
    "than one matches it returns every candidate and no verified id, so ask the customer "
    "for another identifier rather than choosing one. Call this before any order or "
    "refund tool.",
    {"identifier": str},
)
async def get_customer(args):
    raw = str(args["identifier"]).strip()
    needle = raw.lower()
    matches = [c for c in load("customers")
               if needle in (c["customer_id"].lower(), c["email"].lower(),
                             c["phone"].lower(), c["name"].lower())]
    if not matches:
        return failure(
            "validation", "No account matches " + raw + ".", True,
            suggestion="Ask for the email address, phone number or an order number.")
    if len(matches) > 1:
        # A successful query with an ambiguous answer. Returning one "most likely"
        # match here would hide the ambiguity instead of resolving it.
        return ok({
            "matches": len(matches),
            "verified_customer_id": None,
            "candidates": [{"customer_id": c["customer_id"], "name": c["name"],
                            "email_hint": mask(c["email"]),
                            "phone_hint": "*******" + c["phone"][-3:]} for c in matches],
            "next_action": "Ask the customer for an email address, phone number or order "
                           "number before taking any customer-specific action.",
        })
    found = dict(matches[0])
    found["matches"] = 1
    found["verified_customer_id"] = found["customer_id"]
    return ok(found)


@tool(
    "lookup_order",
    "Fetch the full order record for one order id: status, dates, amounts, item and "
    "shipment. Requires a customer id already verified by get_customer. Use it to "
    "establish what was ordered and when it arrived before deciding anything.",
    {"order_id": str, "customer_id": str},
)
async def lookup_order(args):
    order_id = str(args["order_id"]).strip().upper()
    if order_id in FLAKY and order_id not in _timed_out:
        _timed_out.add(order_id)
        return failure("transient", "Timed out after 5000 ms calling the orders API.",
                       True, attempted_query="order_id=" + order_id, partial_results=None)
    for record in load("orders"):
        if record["order_id"] == order_id:
            return ok(record)
    return failure("validation", "No order " + order_id + " on this account.", True,
                   suggestion="Ask the customer to read the order number from their email.")


@tool(
    "process_refund",
    "Refund an amount in pounds against a delivered order, giving a reason. Requires a "
    "verified customer id. Refuses anything outside the refund window. The agent refund "
    "limit is enforced outside this tool, as policy, not as code you can edit here.",
    {"order_id": str, "customer_id": str, "amount": float, "reason": str},
)
async def process_refund(args):
    order_id = str(args["order_id"]).strip().upper()
    record = next((r for r in load("orders") if r["order_id"] == order_id), None)
    if record is None:
        return failure("validation", "No order " + order_id + " on this account.", True,
                       suggestion="Confirm the order number with the customer.")
    delivered = datetime.fromisoformat(record["delivered_at"])
    age = (datetime.now(timezone.utc) - delivered).days
    if age > REFUND_WINDOW_DAYS:
        # A business refusal: never retryable, and it carries language the agent can
        # say out loud to the customer rather than a code it has to translate.
        return failure(
            "business",
            "This order was delivered " + str(age) + " days ago, which is outside the "
            + str(REFUND_WINDOW_DAYS) + " day refund window, so a refund cannot be "
            "processed. A replacement or store credit is available instead, and a human "
            "can approve an exception if the customer needs one.",
            False, policy="refund_window", delivered_days_ago=age,
            window_days=REFUND_WINDOW_DAYS, alternatives=["replacement", "store_credit"])
    return ok({"refund_id": "REF-" + order_id[-4:], "order_id": order_id,
               "amount": round(float(args["amount"]), 2), "state": "processed",
               "reason": args["reason"]})


TRANSCRIPT_REFERENCES = ("as discussed", "as mentioned", "as noted", "see above",
                         "per our conversation", "the above", "earlier in this chat")


class Handoff(BaseModel):
    """What a human who cannot read the transcript needs in order to act."""

    customer_id: str = Field(min_length=3)
    root_cause: str = Field(min_length=10)
    refund_amount: str = Field(min_length=1)
    recommended_action: str = Field(min_length=10)
    issue_summary: str = ""
    order_id: str = ""
    actions_taken: list[str] = Field(default_factory=list)
    escalation_reason: str = ""

    @field_validator("root_cause", "recommended_action", "issue_summary")
    @classmethod
    def no_transcript_references(cls, value):
        lowered = value.lower()
        for phrase in TRANSCRIPT_REFERENCES:
            if phrase in lowered:
                raise ValueError(
                    "the receiving human cannot read the conversation, so "
                    + repr(phrase) + " points at nothing. State the fact itself.")
        return value


HANDOFF_SCHEMA = {
    "type": "object",
    "properties": {
        "customer_id": {"type": "string", "description": "The verified customer id."},
        "root_cause": {"type": "string",
                       "description": "Why this happened, in one or two sentences."},
        "refund_amount": {"type": "string",
                          "description": "The amount at stake, to the penny, or none."},
        "recommended_action": {"type": "string",
                               "description": "What you think the human should do."},
        "issue_summary": {"type": "string", "description": "What the customer asked for."},
        "order_id": {"type": "string", "description": "The order, when there is one."},
        "actions_taken": {"type": "array", "items": {"type": "string"},
                          "description": "What you already tried, in order."},
        "escalation_reason": {"type": "string",
                              "description": "Which escalation trigger fired."},
    },
    "required": ["customer_id", "root_cause", "refund_amount", "recommended_action"],
}


@tool(
    "escalate_to_human",
    "Hand the case to a human agent. The human cannot see this conversation, so the "
    "summary has to stand on its own: who the customer is, what actually went wrong, "
    "the amount at stake, and what you recommend. Call this when the customer asks for "
    "a person, when policy does not cover the request, or when you cannot make progress.",
    HANDOFF_SCHEMA,
)
async def escalate_to_human(args):
    try:
        handoff = Handoff(**args)
    except ValidationError as exc:
        missing = [".".join(str(p) for p in err["loc"]) for err in exc.errors()]
        return failure(
            "validation",
            "The handoff is not self-contained: " + "; ".join(missing) + ".", True,
            suggestion="Resend with a verified customer id, a root cause, the refund "
                       "amount and a recommended action, each stated in full.")
    return ok({"ticket": "ESC-" + handoff.customer_id[-4:], "state": "queued",
               "handoff": handoff.model_dump()})


TOOLS = [get_customer, lookup_order, process_refund, escalate_to_human]
server = create_sdk_mcp_server(name="support", version="1.0.0", tools=TOOLS)
'''

path = SUPPORT / "tools.py"
path.write_text(TOOLS_PY.lstrip(), encoding="utf-8")
print(f"wrote {path.relative_to(WORKSPACE)}  ({len(TOOLS_PY.splitlines())} lines)")

## 3. Call them directly

An SDK tool is not an ordinary function. `@tool` returns an `SdkMcpTool` object, so
`get_customer(...)` raises `TypeError`: it is not callable. The work is on `.handler`, and
`.name`, `.description` and `.input_schema` are exactly what the model is shown. That is the
opposite of Lab 1's FastMCP tools, which stayed ordinary functions you could call directly.

Worth seeing side by side: the single match that verifies, the double match that must not,
the timeout that is worth retrying, and the refusal that is not.

In [ ]:
import importlib
import sys

sys.path.insert(0, str(SUPPORT.parent))
import support.tools as T  # noqa: E402

importlib.reload(T)
T.reset()

print("callable(get_customer):", callable(T.get_customer), " type:", type(T.get_customer).__name__)
print()

for t in T.TOOLS:
    required = t.input_schema.get("required") if isinstance(t.input_schema, dict) else None
    keys = list(t.input_schema.get("properties", {})) if required else list(t.input_schema)
    print(f"  {t.name:20s} {', '.join(keys)}")
    print(f"  {'':20s} {t.description[:88]}...")
print()


async def call(t, **args):
    result = await t.handler(args)
    payload = json.loads(result["content"][0]["text"])
    flag = "is_error" if result.get("is_error") else "ok      "
    return flag, payload


for label, coro in (
    ("one match", call(T.get_customer, identifier="ivan.petrov@example.com")),
    ("two matches", call(T.get_customer, identifier="Sam Okafor")),
    ("no match", call(T.get_customer, identifier="nobody@example.com")),
    ("timeout", call(T.lookup_order, order_id="ORD-0009", customer_id="CUST-0007")),
    ("retried", call(T.lookup_order, order_id="ORD-0009", customer_id="CUST-0007")),
    ("outside window", call(T.process_refund, order_id="ORD-0009", customer_id="CUST-0007",
                            amount=120.0, reason="faulty zip")),
):
    flag, payload = await coro
    summary = payload.get("message") or f"verified_customer_id={payload.get('verified_customer_id')}"
    if payload.get("matches", 0) > 1:
        summary = f"{payload['matches']} candidates, verified_customer_id=None"
    elif "order_id" in payload and "message" not in payload:
        summary = f"{payload['order_id']}, {len(payload)} fields"
    print(f"  {label:16s} {flag}  {str(summary)[:84]}")
print()
print("Two matches came back as a success. The ambiguity is the answer, not a failure.")

## 4. The record is a data problem before it is a reasoning problem

`lookup_order` returns thirty-three fields. Five of them decide a refund. The other
twenty-eight are paid for on every turn they stay in the window, and they are read on every
turn too.

Worse, the two tools disagree about how to write things down. `get_customer` returns Unix
timestamps; `lookup_order` returns ISO 8601 and a numeric status code. Ask the model
whether one order is older than another and you have handed it a formats question instead
of a business question. It will usually get it right. Usually is the same word that
produced the 12%.

Neither of these is fixed by asking the model to be careful.

In [ ]:
flag, customer = await call(T.get_customer, identifier="CUST-0001")
flag, record = await call(T.lookup_order, order_id="ORD-0003", customer_id="CUST-0001")

print("get_customer, the fields that carry a time:")
for key in ("created", "last_contact_at"):
    print(f"  {key:18s} {customer[key]!r}")
print()
print(f"lookup_order, {len(record)} fields, of which the ones a refund turns on:")
for key in ("order_id", "status", "delivered_at", "total_pence", "item_name"):
    print(f"  {key:18s} {record[key]!r}")
print()
print("  and the rest:", ", ".join(k for k in record if k not in
      ("order_id", "status", "delivered_at", "total_pence", "item_name"))[:150], "...")

## 5. One pass fixes both, before the model reads anything

`PostToolUse` sees the result before the model does, and `updatedToolOutput` replaces it.
One callback converts the timestamps, labels the status code and drops the twenty-eight
fields nobody asked for, for every tool matching the pattern, including servers whose code
you do not own.

It cannot un-run a call. That is `PreToolUse`'s job and it is the next two sections.
Transformation is after; prevention is before.

One shape note. An MCP tool answers with an array of typed content blocks, so the JSON a
rule needs is inside a text block rather than sitting at the top of the response. The helper
below reads either shape and puts it back the way it found it.

This same file carries the case facts, because both are answers to one question: what does
the harness remember between calls.

In [ ]:
HOOKS_PY = '''
"""The policy around the tools: what may run, and what the model gets to read."""
import json
from datetime import datetime, timezone

REFUND_LIMIT = 500.0
GATED = ("mcp__support__lookup_order", "mcp__support__process_refund")
STATUS_NAMES = {10: "placed", 20: "dispatched", 30: "delivered", 40: "returned",
                50: "cancelled"}
KEEP = ("order_id", "status", "delivered_at", "total_pence", "item_name")
NEWLINE = chr(10)

# What the harness remembers between calls. The verified id is what the gate checks;
# the issues are the case facts, kept verbatim and never summarised.
case = {"verified_customer_id": None, "customer_name": None, "issues": []}


def reset():
    case.update(verified_customer_id=None, customer_name=None, issues=[])


def payload_of(response):
    """An MCP tool answers with typed content blocks, so the JSON a rule needs is
    inside a text block rather than at the top of the response. Measured against the
    SDK, a hook is handed the block list itself. Read that, the wrapped form, and a
    plain dictionary, and remember which one came in."""
    blocks = None
    shape = None
    if isinstance(response, list):
        blocks, shape = response, "list"
    elif isinstance(response, dict) and isinstance(response.get("content"), list):
        blocks, shape = response["content"], "wrapped"
    if blocks is not None:
        for block in blocks:
            if isinstance(block, dict) and block.get("type") == "text":
                try:
                    return json.loads(block["text"]), shape
                except (ValueError, KeyError, TypeError):
                    return None, None
        return None, None
    if isinstance(response, dict):
        return response, "dict"
    return None, None


def repack(response, payload, shape):
    block = {"type": "text", "text": json.dumps(payload, indent=2)}
    if shape == "list":
        return [block]
    if shape == "wrapped":
        out = dict(response)
        out["content"] = [block]
        return out
    return payload


async def normalise_tool_output(input_data, tool_use_id, context):
    """PostToolUse. Unix to ISO 8601, status code to label, thirty-three fields to five."""
    payload, shape = payload_of(input_data.get("tool_response"))
    if payload is None:
        return {}
    normalised = dict(payload)
    changed = False
    for field in ("created", "last_contact_at"):
        value = normalised.get(field)
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            normalised[field] = datetime.fromtimestamp(value, tz=timezone.utc).isoformat()
            changed = True
    status = normalised.get("status")
    if isinstance(status, int) and not isinstance(status, bool):
        normalised["status"] = STATUS_NAMES.get(status, str(status))
        changed = True
    if input_data.get("tool_name") == "mcp__support__lookup_order" and "total_pence" in normalised:
        trimmed = {key: normalised[key] for key in KEEP if key in normalised}
        trimmed["fields_dropped"] = len(normalised) - len(trimmed)
        normalised, changed = trimmed, True
    if not changed:
        return {}
    return {"hookSpecificOutput": {
        "hookEventName": "PostToolUse",
        "updatedToolOutput": repack(input_data["tool_response"], normalised, shape),
    }}


def note_issue(kind, reference, amount, status):
    """One record per concern, so an amount never attaches to the wrong order."""
    for issue in case["issues"]:
        if issue["reference"] == reference:
            issue.update(amount=amount, status=status)
            return
    case["issues"].append({"kind": kind, "reference": reference, "amount": amount,
                           "status": status})


def render_facts():
    """The block from slide 9.5. Identifiers, amounts and statuses verbatim: a summary
    would blur 40.00 into "about forty pounds" and the wrong order would be refunded."""
    verified = case["verified_customer_id"] or "not yet verified"
    state = "verified" if case["verified_customer_id"] else "unverified"
    lines = ["=== CASE FACTS ===", "CUSTOMER  " + verified + "  " + state]
    for number, issue in enumerate(case["issues"], 1):
        lines.append("ISSUE " + str(number) + "   " + issue["kind"])
        lines.append(("  " + issue["reference"] + "   " + issue["amount"]).rstrip())
        lines.append("  status  " + issue["status"])
    lines.append("===")
    return NEWLINE.join(lines)
'''

path = SUPPORT / "hooks.py"
path.write_text(HOOKS_PY.lstrip(), encoding="utf-8")
print(f"wrote {path.relative_to(WORKSPACE)}")

In [ ]:
import support.hooks as H  # noqa: E402

importlib.reload(H)
H.reset()


async def fire(hook, tool_name, tool_input=None, tool_response=None):
    """Call a hook exactly the way the SDK does: input_data, tool_use_id, context."""
    event = "PostToolUse" if tool_response is not None else "PreToolUse"
    data = {"tool_name": tool_name, "tool_input": tool_input or {},
            "hook_event_name": event}
    if tool_response is not None:
        data["tool_response"] = tool_response
    return await hook(data, "tu_lab3", None)


# A hook is handed the content blocks themselves, which is what the SDK passes.
raw = await T.lookup_order.handler({"order_id": "ORD-0003", "customer_id": "CUST-0001"})
decision = await fire(H.normalise_tool_output, "mcp__support__lookup_order",
                      tool_response=raw["content"])
after = json.loads(decision["hookSpecificOutput"]["updatedToolOutput"][0]["text"])

before = json.loads(raw["content"][0]["text"])
print(f"lookup_order  before {len(before)} fields, after {len(after) - 1} kept"
      f" plus a note that {after['fields_dropped']} were dropped")
print(f"  status        {before['status']!r}  ->  {after['status']!r}")
print(f"  delivered_at  {before['delivered_at'][:19]}  (already ISO, left alone)")
print()

raw_customer = await T.get_customer.handler({"identifier": "CUST-0001"})
decision = await fire(H.normalise_tool_output, "mcp__support__get_customer",
                      tool_response=raw_customer["content"])
customer_after = json.loads(decision["hookSpecificOutput"]["updatedToolOutput"][0]["text"])
print("get_customer")
print(f"  created       {json.loads(raw_customer['content'][0]['text'])['created']!r}"
      f"  ->  {customer_after['created']!r}")
print()

H.case["verified_customer_id"] = customer_after["verified_customer_id"]
H.note_issue("billing", "invoice INV-0002", "40.00", "charged twice, open")
H.note_issue("delivery", "order ORD-0003   SHP-0004", "", "marked delivered, open")
print(H.render_facts())

## 6. The gate that makes ordering a property of the harness

The rule is "verify the customer before any order or refund operation". Written in the
system prompt it holds most of the time. Written as two hooks it holds every time.

`PostToolUse` records a verified id, and only when exactly one account matched.
`PreToolUse` denies the downstream tools until that record exists. Three things make this
the answer rather than a workaround:

- Ordering stops depending on the model's compliance, which means it also holds on the 12%.
- The denial teaches. `permissionDecisionReason` comes back to the agent as the tool result,
  so it reads why it was stopped and calls `get_customer` next. The workflow self-corrects
  rather than erroring out.
- It cannot be bypassed. Hooks are evaluated before the permission mode, so the denial
  stands even under `bypassPermissions`. This is why the gate is a hook and not a mode: we
  measured a mode letting an unlisted tool through, with no denial recorded at all.

Note what the two-match case does here. It records nothing, so the gate stays shut and the
agent has to ask for another identifier. The ambiguity rule and the ordering rule turn out
to be the same mechanism.

In [ ]:
GATE_PY = '''

async def record_verification(input_data, tool_use_id, context):
    """PostToolUse. Remember the verified id, and only on exactly one match."""
    if input_data.get("tool_name") != "mcp__support__get_customer":
        return {}
    payload, _ = payload_of(input_data.get("tool_response"))
    if payload and payload.get("matches") == 1 and payload.get("verified_customer_id"):
        case["verified_customer_id"] = payload["verified_customer_id"]
        case["customer_name"] = payload.get("name")
    return {}


async def require_verification(input_data, tool_use_id, context):
    """PreToolUse. Deny the downstream tools until the prerequisite has returned."""
    if input_data.get("tool_name") in GATED and not case["verified_customer_id"]:
        return {"hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": (
                "Verify the customer with get_customer before any order or refund "
                "operation. If more than one account matched, ask the customer for "
                "another identifier first, then verify."),
        }}
    return {}


async def enforce_refund_limit(input_data, tool_use_id, context):
    """PreToolUse. Block, and redirect by naming the workflow that can proceed."""
    if input_data.get("tool_name") != "mcp__support__process_refund":
        return {}
    try:
        amount = float(input_data.get("tool_input", {}).get("amount") or 0)
    except (TypeError, ValueError):
        amount = 0.0
    if amount > REFUND_LIMIT:
        return {"hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": (
                "A refund of " + format(amount, ".2f") + " pounds is over the "
                + format(REFUND_LIMIT, ".0f") + " pound agent limit and needs a human. "
                "Call escalate_to_human with the customer id, the root cause, the "
                "amount and a recommended action."),
        }}
    return {}
'''

path = SUPPORT / "hooks.py"
path.write_text(path.read_text(encoding="utf-8") + GATE_PY, encoding="utf-8")
importlib.reload(H)
H.reset()

verified = await T.get_customer.handler({"identifier": "ivan.petrov@example.com"})
ambiguous = await T.get_customer.handler({"identifier": "Sam Okafor"})
refund_args = {"order_id": "ORD-0003", "customer_id": "CUST-0001", "amount": 89.99,
               "reason": "damaged on arrival"}


async def verdict(tool_name, tool_input=None):
    decision = await fire(H.require_verification, tool_name, tool_input=tool_input)
    if not decision:
        return "allowed"
    return "DENIED: " + decision["hookSpecificOutput"]["permissionDecisionReason"][:58]


print("before any verification")
print("  process_refund   ", await verdict("mcp__support__process_refund", refund_args))
print()

await fire(H.record_verification, "mcp__support__get_customer",
           tool_response=ambiguous["content"])
print("after get_customer returned two candidates")
print("  recorded id      ", H.case["verified_customer_id"])
print("  process_refund   ", await verdict("mcp__support__process_refund", refund_args))
print()

await fire(H.record_verification, "mcp__support__get_customer",
           tool_response=verified["content"])
print("after get_customer returned exactly one")
print("  recorded id      ", H.case["verified_customer_id"])
print("  process_refund   ", await verdict("mcp__support__process_refund", refund_args))

## 7. The threshold, and why a denial names its alternative

Policy says an agent may refund up to 500 pounds. `process_refund` does not enforce that,
on purpose: the tool executes what it is told, and the limit lives in the harness where it
applies to every caller and can change without redeploying the backend. Part B watches a
650 pound refund go through when the hook is not loaded.

The whole lesson is in the reason string. A bare refusal leaves the agent guessing, and a
guessing agent either retries the same call or tells the customer nothing useful. Naming
`escalate_to_human` is what turns a refusal into a redirect: the policy violation becomes a
different workflow instead of a dead end.

In [ ]:
for amount in (89.99, 500.00, 650.00):
    decision = await fire(H.enforce_refund_limit, "mcp__support__process_refund",
                          tool_input={**refund_args, "amount": amount})
    if decision:
        specific = decision["hookSpecificOutput"]
        print(f"  {amount:7.2f}  {specific['permissionDecision']}")
        for line in specific["permissionDecisionReason"].split(". "):
            print(f"           {line.strip()}")
    else:
        print(f"  {amount:7.2f}  allowed")
print()
print("A hook returning {} is the way to say: no opinion, carry on.")

## 8. A handoff the receiving human can act on

Escalation crosses a context boundary. The human who picks the case up **cannot read this
conversation**; they see only what the agent hands over. So the handoff is not a summary of
a chat, it is a record that stands alone.

The exam's minimum set is four fields: customer id, root cause, refund amount, recommended
action. The guide's fuller version adds the name, the issue summary, the order, what was
already tried and which trigger fired. None of them contradict, and all of them earn their
place: `escalation_reason` routes the case to a queue, `refund_amount` decides who has the
authority to approve it, and `actions_taken` stops a human re-offering the replacement the
customer has already refused.

Structure is what makes this checkable. A schema can refuse a handoff that forgot the
amount. A paragraph cannot be checked for the field it left out, and it cannot be checked
for pointing at a conversation the reader does not have.

In [ ]:
cases = {
    "a prose summary": {
        "customer_id": "CUST-0001",
        "root_cause": "The customer is unhappy, as discussed above, and wants a refund.",
        "refund_amount": "650.00",
        "recommended_action": "Please take a look and sort it out for them.",
    },
    "missing the amount": {
        "customer_id": "CUST-0001",
        "root_cause": "Espresso machine arrived with a cracked housing; photos supplied.",
        "recommended_action": "Approve the full refund; the fault is not in dispute.",
    },
    "self-contained": {
        "customer_id": "CUST-0001",
        "issue_summary": "Refund request for a damaged espresso machine.",
        "order_id": "ORD-0021",
        "root_cause": "Espresso machine arrived with a cracked housing; photos supplied.",
        "actions_taken": ["Verified the customer with get_customer",
                          "Confirmed order ORD-0021 with lookup_order",
                          "Offered a replacement; the customer declined it"],
        "refund_amount": "650.00",
        "recommended_action": "Approve the full refund of 650.00 pounds.",
        "escalation_reason": "Above the 500 pound agent refund limit.",
    },
}

for label, payload in cases.items():
    result = await T.escalate_to_human.handler(payload)
    body = json.loads(result["content"][0]["text"])
    if result.get("is_error"):
        print(f"  {label:20s} REFUSED  {body['message']}")
        print(f"  {'':20s}          {body['suggestion'][:70]}...")
    else:
        print(f"  {label:20s} queued as {body['ticket']}")
print()
print("The first one carries all four fields and is still refused: 'as discussed above'")
print("points at a transcript the receiving human does not have.")

## What part A built

| Diagram box | Where it was built |
|---|---|
| In-process MCP tools, four of them | Section 2 |
| transient retry, business refusal | Sections 2 and 3 |
| PostToolUse normaliser | Section 5 |
| PreToolUse policy, prerequisite | Section 6 |
| PreToolUse policy, threshold | Section 7 |
| Human handoff, self-contained | Section 8 |

Everything above is deterministic, and that is the point worth carrying out of this part.
None of it asked a model for anything, so none of it has a failure rate. The gates hold on
every run, including the run where the model would have got it wrong.

What is left is the half that cannot be guaranteed: whether to resolve or escalate, and
whether to ask or assume. Part B adds the agent, runs six conversations at it, and reports
that half honestly, as a rate.